In [ ]:
import numpy as np
import pandas as pd
from scipy import signal
from matplotlib import pyplot as plt
import seaborn as sns
import os

# Suppress FutureWarnings raised by the pandas .interpolate() method
import warnings
warnings.simplefilter('ignore', FutureWarning)

In [ ]:
# --- Load CSV ---
data_dir = "./data"
output_dir = "./output/lpf"
ver = '20260618'

df = pd.read_csv(os.path.join(data_dir, f"{ver}_cvp.csv"))

# --- Compute the offset in minutes ---
# Convert time and main_start_time to datetime
df['time'] = pd.to_datetime(df['time'])
df['main_start_time'] = pd.to_datetime(df['main_start_time'])

# Compute the time difference in minutes
df['offset'] = (df['time'] - df['main_start_time']).dt.total_seconds() / 60
df['offset'] = df['offset'].astype(int)  # Convert to integer, assuming a 1-minute sampling interval

# Keep only the required columns
df = df[['icu_stay_id', 'active_ingredient_name', 'offset', 'cvp']]


# --- Fill in missing offsets ---
def expand_group(group):
    icu_stay_id = group['icu_stay_id'].iloc[0]
    active_ingredient_name = group['active_ingredient_name'].iloc[0]
    min_offset = group['offset'].min()
    max_offset = group['offset'].max()
    full_range = pd.DataFrame({
        'icu_stay_id': icu_stay_id,
        'active_ingredient_name': active_ingredient_name,
        'offset': np.arange(min_offset, max_offset + 1)
    })
    return full_range.merge(group, on=['icu_stay_id', 'active_ingredient_name', 'offset'], how='left')

df = df.groupby(['icu_stay_id', 'active_ingredient_name'], group_keys=False).apply(expand_group)
df = df.sort_values(by=['icu_stay_id', 'active_ingredient_name', 'offset']).reset_index(drop=True)

len(df), len(df['icu_stay_id'].unique())


In [ ]:
df.head()

In [ ]:
def is_continuous(offset_series):
    return (offset_series.sort_values().diff().dropna() == 1).all()

# Check continuity for each icu_stay_id
continuity_check = df.groupby(['icu_stay_id', 'active_ingredient_name'])['offset'].apply(is_continuous)

# Show ICU stay IDs where offset is not continuous
non_continuous_ids = continuity_check[~continuity_check].index.tolist()

len(non_continuous_ids)

In [ ]:
pid = df['icu_stay_id'].unique()[0]
drug = df[df['icu_stay_id'] == pid]['active_ingredient_name'].iloc[0]

d = (
    df[(df['icu_stay_id'] == pid) &
       (df['active_ingredient_name'] == drug)]
    .sort_values('offset')
)

plt.figure(figsize=(10, 6))
plt.plot(d['offset'], d['cvp'])
plt.xlabel('Offset (minutes)')
plt.ylabel('CVP')
plt.title(f'Raw CVP (ICU {pid}, {drug})')
plt.show()


In [ ]:
def lowpass(x, samplerate, fp, fs, gpass, gstop):
    fn = samplerate / 2   # Nyquist frequency
    wp = fp / fn  # Normalize the passband edge frequency
    ws = fs / fn  # Normalize the stopband edge frequency
    N, Wn = signal.buttord(wp, ws, gpass, gstop)  # Compute the normalized Butterworth frequency
    b, a = signal.butter(N, Wn, "low")            # Compute the numerator and denominator of the filter transfer function
    y = signal.filtfilt(b, a, x)                  # Apply the filter
    return y

In [ ]:
samplerate = 1/60
fpass = 0.15*samplerate # Passband edge frequency [Hz]
fstop = 0.40*samplerate # Stopband edge frequency [Hz]
gpass = 3 # Maximum loss at the passband edge [dB]; shapes the attenuation curve between fpass and fstop
gstop = 40 # Minimum attenuation at the stopband edge [dB]; shapes the attenuation curve between fpass and fstop

df = df.sort_values(
    ['icu_stay_id', 'active_ingredient_name', 'offset']
).reset_index(drop=True)

df['cvp_lpf'] = np.nan

for (icu_stay_id, active_ingredient_name), df_tmp in df.groupby(
    ['icu_stay_id', 'active_ingredient_name']
):
    df_tmp = df_tmp.sort_values('offset').copy()

    # Interpolate (within each drug only)
    df_tmp['cvp'] = df_tmp['cvp'].interpolate(limit_direction='both')

    x = df_tmp['cvp'].values

    # Check the minimum number of data points
    if len(x) <= 12:
        continue

    y = lowpass(x, samplerate, fpass, fstop, gpass, gstop)

    # Write back to the original data frame
    df.loc[df_tmp.index, 'cvp_lpf'] = y


In [ ]:
pairs = (
    df[['icu_stay_id', 'active_ingredient_name']]
    .drop_duplicates()
    .reset_index(drop=True)
)

for i in range(10):
    icu_stay_id = pairs.loc[i + 10, 'icu_stay_id']
    active_ingredient_name = pairs.loc[i + 10, 'active_ingredient_name']

    d = (
        df[(df['icu_stay_id'] == icu_stay_id) &
           (df['active_ingredient_name'] == active_ingredient_name)]
        .sort_values('offset')
    )

    plt.figure(figsize=(10, 6))
    plt.plot(d['offset'], d['cvp'], label='RAW cvp')
    plt.plot(d['offset'], d['cvp_lpf'], label='FILTERED cvp')

    plt.xlabel('Offset (minutes)')
    plt.ylabel('CVP')
    plt.title(f'CVP LPF (ICU {icu_stay_id}, {active_ingredient_name})')
    plt.legend()
    plt.show()


In [ ]:
df[['cvp', 'cvp_lpf']].isnull().sum()

In [ ]:
df = df.dropna(subset=['cvp_lpf']).copy()

In [ ]:
len(df), len(df['icu_stay_id'].unique())

In [ ]:
df.to_csv(os.path.join(data_dir, f"{ver}_cvp_filtered.csv"), header=True, index=False)


In [ ]:
patient_id = df['icu_stay_id'].unique()[20]
active_ingredient_name = (
    df[df['icu_stay_id'] == patient_id]['active_ingredient_name']
    .iloc[0]
)

patient_data = (
    df[(df['icu_stay_id'] == patient_id) &
       (df['active_ingredient_name'] == active_ingredient_name)]
    .sort_values('offset')
)

plot_df = patient_data[['offset', 'cvp', 'cvp_lpf']].melt(
    id_vars='offset',
    var_name='Signal Type',
    value_name='cvp Value'
)

plot_df['Signal Type'] = plot_df['Signal Type'].map({
    'cvp': 'Raw cvp',
    'cvp_lpf': 'Filtered cvp'
})

palette = {'Raw cvp': '#97BBF5CC', 'Filtered cvp': '#4269D0'}

fig, ax = plt.subplots(figsize=(12, 8))
sns.lineplot(
    data=plot_df,
    x='offset',
    y='cvp Value',
    hue='Signal Type',
    palette=palette,
    linewidth=2.5,
    ax=ax
)

ax.set_xticks([-30, 0, 30, 60, 90, 120])
ax.set_title(f'Raw vs LPF CVP (ICU {patient_id}, {active_ingredient_name})', fontsize=24)
ax.set_xlabel('Time Offset (minutes)', fontsize=20)
ax.set_ylabel('Central Vein Pressure (mmHg)', fontsize=20)
ax.tick_params(axis='both', which='major', labelsize=18)
ax.legend(title='Signal Type', fontsize=16, title_fontsize=18)

plt.tight_layout()
plt.show()
